In [1]:
import os
import random
import torch
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [10]:
from config import (
    SAMPLE_RATE,
    N_FFT,
    HOP_LENGTH,
    WIN_LENGTH,
    WINDOW_TYPE,
    FIXED_TIME_FRAMES,
    USE_LOG_MAG,
    PROCESSED_ROOT,
    PROCESSED_DIR,
    PROCESSED_AUDIO_DIR,
    CLEAN_EN_DIR,
    CLEAN_NP_DIR,
    CLEAN_NP_WEBM_DIR,
    CLEAN_EN_MP3_DIR,
    NOISE_DIR,
    NOISY_PAIRS_PER_CLEAN,
    SNR_LIST,
    FFMPEG_PATH,
)

# ================= CONFIG =================
WINDOW = torch.hann_window(WIN_LENGTH)

# Create processed directories
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(PROCESSED_DIR, split, "clean"), exist_ok=True)
    os.makedirs(os.path.join(PROCESSED_DIR, split, "noisy"), exist_ok=True)


In [3]:
FOLDER = "dataset/raw/clean/clean_en"
TRIM_SECONDS = 0.5


for filename in os.listdir(FOLDER):
    
    if filename.startswith("en") and filename.lower().endswith((".wav", ".mp3", ".flac")):
        
        file_path = os.path.join(FOLDER, filename)

        y, sr = librosa.load(file_path, sr=None)
        trim_samples = int(TRIM_SECONDS * sr)

        if len(y) > 2 * trim_samples:
            trimmed_audio = y[trim_samples: len(y) - trim_samples]
            
            # Overwrite original file
            sf.write(file_path, trimmed_audio, sr)
            print(f"Replaced: {filename}")
        else:
            print(f"Skipped (too short): {filename}")

print("Done.")

Replaced: en_36189601.wav
Replaced: en_36189609.wav
Replaced: en_36189655.wav
Replaced: en_36189663.wav
Replaced: en_36189665.wav
Replaced: en_36189926.wav
Replaced: en_36189974.wav
Replaced: en_36190815.wav
Replaced: en_36190839.wav
Replaced: en_36191023.wav
Replaced: en_36191026.wav
Replaced: en_36197259.wav
Replaced: en_36197271.wav
Replaced: en_36197279.wav
Replaced: en_36197362.wav
Replaced: en_36198950.wav
Replaced: en_36198975.wav
Replaced: en_36199976.wav
Replaced: en_36200025.wav
Replaced: en_36200059.wav
Replaced: en_36200077.wav
Replaced: en_36200800.wav
Replaced: en_36200825.wav
Replaced: en_36209401.wav
Replaced: en_36209528.wav
Replaced: en_36209573.wav
Replaced: en_36209889.wav
Replaced: en_36209897.wav
Replaced: en_36210102.wav
Replaced: en_36211463.wav
Replaced: en_36211481.wav
Replaced: en_36211487.wav
Replaced: en_36211515.wav
Replaced: en_36211557.wav
Replaced: en_36211605.wav
Replaced: en_36230850.wav
Replaced: en_36230859.wav
Replaced: en_36230917.wav
Replaced: en

In [ ]:
import subprocess

FFMPEG = FFMPEG_PATH

src_dir = CLEAN_NP_WEBM_DIR
dst_dir = CLEAN_NP_DIR
os.makedirs(dst_dir, exist_ok=True)

# Check if directory exists before listing
if os.path.exists(src_dir):
    for file in os.listdir(src_dir):
        if file.endswith(".webm"):
            src_path = os.path.join(src_dir, file)
            dst_path = os.path.join(dst_dir, file.replace(".webm", ".wav"))

            command = [
                FFMPEG, "-y",
                "-i", src_path,
                "-ar", str(SAMPLE_RATE),
                dst_path
            ]

            subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    print("✅ All files converted to 16 kHz WAV")
else:
    print(f"Skipping Windows conversion: {src_dir} not found (likely on Colab/Linux)")


In [ ]:
# For Linux

# Paths
SRC_DIR_NP = CLEAN_NP_WEBM_DIR   # source .webm files
SRC_DIR_EN = CLEAN_EN_MP3_DIR    # source .mp3 files
DST_DIR_NP = CLEAN_NP_DIR
DST_DIR_EN = CLEAN_EN_DIR

os.makedirs(DST_DIR_NP, exist_ok=True)
os.makedirs(DST_DIR_EN, exist_ok=True)

# List all .webm files
if os.path.exists(SRC_DIR_NP):
    webm_files = [f for f in os.listdir(SRC_DIR_NP) if f.lower().endswith(".webm")]
    print(f"Found {len(webm_files)} .webm files to convert...")

    # Convert each .webm to .wav
    for f in webm_files:
        src_path = os.path.join(SRC_DIR_NP, f)
        dst_path = os.path.join(DST_DIR_NP, os.path.splitext(f)[0] + ".wav")
        command = [FFMPEG_PATH, "-y", "-i", src_path, "-ar", str(SAMPLE_RATE), dst_path]
        subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("✅ Conversion complete. All files saved in clean_np as .wav")
else:
    print(f"Skipping webm conversion: {SRC_DIR_NP} not found")

# List all .mp3 files
if os.path.exists(SRC_DIR_EN):
    mp3_files = [f for f in os.listdir(SRC_DIR_EN) if f.lower().endswith(".mp3")]
    print(f"Found {len(mp3_files)} .mp3 files to convert...")

    for f in mp3_files:
        src_path = os.path.join(SRC_DIR_EN, f)
        dst_path = os.path.join(DST_DIR_EN, os.path.splitext(f)[0] + ".wav")
        command = [FFMPEG_PATH, "-y", "-i", src_path, "-ar", str(SAMPLE_RATE), dst_path]
        subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE )

    print("Conversion Complete. All files saved in clean_en as .wav")
else:
    print(f"Skipping mp3 conversion: {SRC_DIR_EN} not found")


Found 1315 .webm files to convert...
Found 1000 .mp3 files to convert...
Conversion Complete. All files saved in clean_en as .wav


In [15]:
def collect_files(directory, extensions=(".wav", ".mp3")):
    """Recursively collect files from directory and subfolders."""
    files = []
    for root, _, filenames in os.walk(directory):
        for f in filenames:
            if f.lower().endswith(extensions):
                files.append(os.path.join(root, f))
    return sorted(files)


def load_audio(path):
    """Load audio as 1D numpy array at SAMPLE_RATE."""
    audio, _ = librosa.load(path, sr=SAMPLE_RATE)
    return audio


def fix_length(audio, target_len=SAMPLE_RATE * 4):
    """Pad or trim audio to fixed length (default 4s).
    
    If shorter than target_len, the audio is looped (repeated) to fill the duration.
    This prevents 'silence padding' which wastes training capacity.
    """
    if len(audio) < target_len:
        # Loop the audio until it exceeds target_len
        n_repeats = int(np.ceil(target_len / len(audio)))
        audio = np.tile(audio, n_repeats)
    
    # Trim to exact target length
    audio = audio[:target_len]
    return audio


def add_noise(clean_audio, noise_audio, snr_db=5):
    """Mix noise with clean audio at given SNR."""
    noise_audio = noise_audio[:len(clean_audio)]
    clean_power = np.mean(clean_audio ** 2)
    noise_power = np.mean(noise_audio ** 2)
    target_noise_power = clean_power / (10 ** (snr_db / 10))
    noise_audio = noise_audio * np.sqrt(target_noise_power / (noise_power + 1e-8))
    return clean_audio + noise_audio

def compute_stft(audio):
    """Compute STFT magnitude and phase tensors (513 bins).

    Returns (mag, phase) where phase is angle radians; no log applied here.
    """
    waveform = torch.tensor(audio, dtype=torch.float32)
    stft = torch.stft(
        waveform,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=WINDOW,
        return_complex=True,
    )
    mag = torch.abs(stft)
    phase = torch.angle(stft)
    return mag, phase  # [513, T]


In [ ]:
# Collect files
clean_en_files = collect_files(CLEAN_EN_DIR, (".wav",))
clean_np_files = collect_files(CLEAN_NP_DIR, (".wav",))

# Randomly select 1200 audios
random.seed(42)  # for reproducibility

num_en_to_use = int(len(clean_en_files) * 0.75)
num_np_to_use = int(len(clean_np_files) * 0.60)

selected_en = random.sample(clean_en_files, num_en_to_use)
selected_np = random.sample(clean_np_files, num_np_to_use)

# Combine them with language labels
en_labeled = [(f, "en") for f in selected_en]
np_labeled = [(f, "np") for f in selected_np]
clean_files = en_labeled + np_labeled

print(f"English files selected: {len(selected_en)} (from {len(clean_en_files)})")
print(f"Nepali files selected: {len(selected_np)} (from {len(clean_np_files)})")
print(f"Total clean files selected: {len(clean_files)}")

# Noise 
noise_files = collect_files(NOISE_DIR, (".wav",))
if len(noise_files) == 0:
    raise ValueError("No noise files found in NOISE_DIR and subfolders!")


# Split 6:2:2 (adjusting test_size based on actual total count)
# To handle variable total counts, we use ratios for splitting instead of fixed numbers if the total size changes significantly.
# Original logic was fixed numbers (720 train, 240 val, 240 test = 1200 total).
# If correct total is different, we should probably stick to proportional splits.

# Let's use proportional split: 60% Train, 20% Val, 20% Test
train_files, test_val_files = train_test_split(clean_files, test_size=0.4, random_state=42)
val_files, test_files = train_test_split(test_val_files, test_size=0.5, random_state=42)

print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

English files selected: 1318 (from 1758)
Nepali files selected: 789 (from 1315)
Total clean files selected: 2107
Train: 1264 | Val: 421 | Test: 422


In [16]:
def preprocess_and_save(clean_list, save_clean_dir, save_noisy_dir):
    idx_counter = 0
    for clean_path, lang in clean_list:
        clean_audio = load_audio(clean_path)
        clean_audio = fix_length(clean_audio)
        clean_mag, clean_phase = compute_stft(clean_audio)

        for pair_idx in range(NOISY_PAIRS_PER_CLEAN):
            # Noisy A
            noise_path_a = random.choice(noise_files)
            noise_audio_a = load_audio(noise_path_a)
            noise_audio_a = fix_length(noise_audio_a)
            snr_a = random.choice(SNR_LIST)
            noisy_audio_a = add_noise(clean_audio, noise_audio_a, snr_db=snr_a)
            noisy_mag_a, noisy_phase_a = compute_stft(noisy_audio_a)

            # Noisy B (different noise clip to encourage independence)
            noise_path_b = random.choice(noise_files)
            while noise_path_b == noise_path_a and len(noise_files) > 1:
                noise_path_b = random.choice(noise_files)
            noise_audio_b = load_audio(noise_path_b)
            noise_audio_b = fix_length(noise_audio_b)
            snr_b = random.choice(SNR_LIST)
            noisy_audio_b = add_noise(clean_audio, noise_audio_b, snr_db=snr_b)
            noisy_mag_b, noisy_phase_b = compute_stft(noisy_audio_b)

            clean_save_path = os.path.join(save_clean_dir, f"{idx_counter:05d}.pt")
            noisy_save_path = os.path.join(save_noisy_dir, f"{idx_counter:05d}.pt")

            # Save clean for evaluation/visualization; noisy contains paired variants for Noise2Noise
            torch.save((clean_mag, clean_phase), clean_save_path)
            torch.save(
                {
                    "a_mag": noisy_mag_a,
                    "a_phase": noisy_phase_a,
                    "b_mag": noisy_mag_b,
                    "b_phase": noisy_phase_b,
                },
                noisy_save_path,
            )

            idx_counter += 1

        if idx_counter % 10 == 0:
            print(f"Processed {idx_counter} pairs...")

    return idx_counter

train_count = preprocess_and_save(train_files, os.path.join(PROCESSED_DIR,"train/clean"), os.path.join(PROCESSED_DIR,"train/noisy"))
print(f"Saved {train_count} training pairs.")

val_count = preprocess_and_save(val_files, os.path.join(PROCESSED_DIR,"val/clean"), os.path.join(PROCESSED_DIR,"val/noisy"))
print(f"Saved {val_count} validation pairs.")

test_count = preprocess_and_save(test_files, os.path.join(PROCESSED_DIR,"test/clean"), os.path.join(PROCESSED_DIR,"test/noisy"))
print(f"Saved {test_count} test pairs.")

print("✅ Preprocessing complete. All splits saved with Noise2Noise pairs.")

Processed 30 pairs...
Processed 60 pairs...
Processed 90 pairs...
Processed 120 pairs...
Processed 150 pairs...
Processed 180 pairs...
Processed 210 pairs...
Processed 240 pairs...
Processed 270 pairs...
Processed 300 pairs...
Processed 330 pairs...
Processed 360 pairs...
Processed 390 pairs...
Processed 420 pairs...
Processed 450 pairs...
Processed 480 pairs...
Processed 510 pairs...
Processed 540 pairs...
Processed 570 pairs...
Processed 600 pairs...
Processed 630 pairs...
Processed 660 pairs...
Processed 690 pairs...
Processed 720 pairs...
Processed 750 pairs...
Processed 780 pairs...
Processed 810 pairs...
Processed 840 pairs...
Processed 870 pairs...
Processed 900 pairs...
Processed 930 pairs...
Processed 960 pairs...
Processed 990 pairs...
Processed 1020 pairs...
Processed 1050 pairs...
Processed 1080 pairs...
Processed 1110 pairs...
Processed 1140 pairs...
Processed 1170 pairs...
Processed 1200 pairs...
Processed 1230 pairs...
Processed 1260 pairs...
Processed 1290 pairs...
Proc

In [ ]:
# Save noisy audio as .wav in dataset/processed_audio (Colab/Drive paths from config)
# Prepare directories for wav exports
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(PROCESSED_AUDIO_DIR, split, "clean"), exist_ok=True)
    os.makedirs(os.path.join(PROCESSED_AUDIO_DIR, split, "noisy"), exist_ok=True)


def preprocess_and_save_with_wav(clean_list, split):
    """Preprocess, save STFT tensors (Double-Noisy for N2N), and export wavs."""
    save_clean_dir = os.path.join(PROCESSED_DIR, split, "clean")
    save_noisy_dir = os.path.join(PROCESSED_DIR, split, "noisy")
    clean_wav_dir = os.path.join(PROCESSED_AUDIO_DIR, split, "clean")
    noisy_wav_dir = os.path.join(PROCESSED_AUDIO_DIR, split, "noisy")
    
    # Ensure all directories exist
    os.makedirs(save_clean_dir, exist_ok=True)
    os.makedirs(save_noisy_dir, exist_ok=True)
    os.makedirs(clean_wav_dir, exist_ok=True)
    os.makedirs(noisy_wav_dir, exist_ok=True)

    idx_counter = 0
    for clean_path, lang in clean_list:
        clean_audio = load_audio(clean_path)
        clean_audio = fix_length(clean_audio)
        clean_mag, clean_phase = compute_stft(clean_audio)

        # Iterate NOISY_PAIRS_PER_CLEAN times to generate multiple samples from one clean file
        for pair_idx in range(NOISY_PAIRS_PER_CLEAN):
            # Generate TWO independent noise realizations per sample for Noise2Noise
            
            # Realization A (Input)
            noise_path_a = random.choice(noise_files)
            noise_audio_a = load_audio(noise_path_a)
            noise_audio_a = fix_length(noise_audio_a)
            snr_a = random.choice(SNR_LIST)
            noisy_audio_a = add_noise(clean_audio, noise_audio_a, snr_db=snr_a)
            noisy_mag_a, noisy_phase_a = compute_stft(noisy_audio_a)
            
            # Realization B (Target)
            noise_path_b = random.choice(noise_files)
            noise_audio_b = load_audio(noise_path_b)
            noise_audio_b = fix_length(noise_audio_b)
            snr_b = random.choice(SNR_LIST)
            noisy_audio_b = add_noise(clean_audio, noise_audio_b, snr_db=snr_b)
            noisy_mag_b, noisy_phase_b = compute_stft(noisy_audio_b)

            # Save Tensors: Clean, plus BOTH noisy versions
            clean_save_path = os.path.join(save_clean_dir, f"{idx_counter:05d}.pt")
            noisy_save_path = os.path.join(save_noisy_dir, f"{idx_counter:05d}.pt")
            
            # Save clean as usual
            torch.save((clean_mag, clean_phase), clean_save_path)
            # Save tuple of 4 for N2N (or just noisy A if standard, but we save 4 to be flexible)
            torch.save((noisy_mag_a, noisy_phase_a, noisy_mag_b, noisy_phase_b), noisy_save_path)

            # Export WAVs (A is strictly input, B is target)
            snr_tag_a = str(snr_a).replace(".", "p").replace("-", "m")
            clean_wav_path = os.path.join(clean_wav_dir, f"{idx_counter:05d}.wav")
            noisy_wav_path_a = os.path.join(noisy_wav_dir, f"{idx_counter:05d}_A_snr{snr_tag_a}.wav")
            
            sf.write(clean_wav_path, clean_audio, SAMPLE_RATE)
            sf.write(noisy_wav_path_a, noisy_audio_a, SAMPLE_RATE)
            # We don't necessarily save wav B unless debugging, to save space

            idx_counter += 1

        if idx_counter % 10 == 0:
            print(f"Processed {idx_counter} pairs for {split}...")

    return idx_counter


# Run this instead of preprocess_and_save if you want .wav outputs
train_wav_count = preprocess_and_save_with_wav(train_files, "train")
val_wav_count = preprocess_and_save_with_wav(val_files, "val")
test_wav_count = preprocess_and_save_with_wav(test_files, "test")
print(f"Saved WAVs: train={train_wav_count}, val={val_wav_count}, test={test_wav_count}")


Processed 30 pairs for train...
Processed 60 pairs for train...
Processed 90 pairs for train...
Processed 120 pairs for train...
Processed 150 pairs for train...
Processed 180 pairs for train...
Processed 210 pairs for train...
Processed 240 pairs for train...
Processed 270 pairs for train...
Processed 300 pairs for train...
Processed 330 pairs for train...
Processed 360 pairs for train...
Processed 390 pairs for train...
Processed 420 pairs for train...
Processed 450 pairs for train...
Processed 480 pairs for train...
Processed 510 pairs for train...
Processed 540 pairs for train...
Processed 570 pairs for train...
Processed 600 pairs for train...
Processed 630 pairs for train...
Processed 660 pairs for train...
Processed 690 pairs for train...
Processed 720 pairs for train...
Processed 750 pairs for train...
Processed 780 pairs for train...
Processed 810 pairs for train...
Processed 840 pairs for train...
Processed 870 pairs for train...
Processed 900 pairs for train...
Processed 930

In [17]:
from config import SAMPLE_RATE

def calculate_snr(clean_signal, noisy_signal):
    """
    Calculate the Signal-to-Noise Ratio (SNR) in dB.
    SNR = 10 * log10(P_signal / P_noise)
    """
    signal_power = np.mean(clean_signal ** 2)
    noise_power = np.mean((clean_signal - noisy_signal) ** 2)
    if noise_power == 0:
        return float('inf')  # Infinite SNR if no noise
    return 10 * np.log10(signal_power / noise_power)

def analyze_snr_distribution(clean_dir, noisy_dir):
    """
    Analyze the SNR distribution for a dataset.
    Args:
        clean_dir (str): Path to the directory containing clean audio files.
        noisy_dir (str): Path to the directory containing noisy audio files.
    """
    snr_values = []

    # Ensure both directories exist
    if not os.path.exists(clean_dir) or not os.path.exists(noisy_dir):
        print("Error: One or both directories do not exist.")
        return

    # Iterate through clean audio files
    for file_name in os.listdir(clean_dir):
        clean_path = os.path.join(clean_dir, file_name)
        noisy_path = os.path.join(noisy_dir, file_name)

        # Check if corresponding noisy file exists
        if not os.path.exists(noisy_path):
            print(f"Warning: No matching noisy file for {file_name}. Skipping...")
            continue

        # Load clean and noisy audio
        clean_signal, sr_clean = sf.read(clean_path)
        noisy_signal, sr_noisy = sf.read(noisy_path)

        # Ensure sampling rates match
        if sr_clean != SAMPLE_RATE or sr_noisy != SAMPLE_RATE:
            print(f"Error: Sampling rates do not match for {file_name}. Skipping...")
            continue

        # Calculate SNR
        snr = calculate_snr(clean_signal, noisy_signal)
        snr_values.append(snr)

    # Analyze SNR distribution
    if snr_values:
        print(f"Number of samples analyzed: {len(snr_values)}")
        print(f"Average SNR: {np.mean(snr_values):.2f} dB")
        print(f"Median SNR: {np.median(snr_values):.2f} dB")
        print(f"Min SNR: {np.min(snr_values):.2f} dB")
        print(f"Max SNR: {np.max(snr_values):.2f} dB")

        # Plot histogram
        plt.hist(snr_values, bins=20, color='blue', edgecolor='black', alpha=0.7)
        plt.title("SNR Distribution")
        plt.xlabel("SNR (dB)")
        plt.ylabel("Frequency")
        plt.grid(True)
        plt.show()
    else:
        print("No valid SNR values calculated. Please check your dataset.")

# Example usage
clean_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "val/clean")  # Path to clean audio files
noisy_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "val/noisy")  # Path to noisy audio files

analyze_snr_distribution(clean_audio_dir, noisy_audio_dir)

No valid SNR values calculated. Please check your dataset.


In [ ]:
# Analyze SNR for test files
clean_test_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "test/clean")  # Path to clean test audio files
noisy_test_audio_dir = os.path.join(PROCESSED_AUDIO_DIR, "test/noisy")  # Path to noisy test audio files

analyze_snr_distribution(clean_test_audio_dir, noisy_test_audio_dir)